# Разбор A. Проверяем запись истрадефиллина

Перед запуском прочитайте [постановку](../../../../../../theory/modules/M03/lessons/L01/01_problem.md), [модель](../../../../../../theory/modules/M03/lessons/L01/02_model.md) и [процедуру](../../../../../../theory/modules/M03/lessons/L01/03_workflow.md). Здесь повторим решение в исполняемых шагах. Отчёт строим из наблюдаемых данных, а не из имени молекулы.

In [2]:
from pathlib import Path
import sys, json, math
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Path.cwd() — текущая папка; parents позволяет найти корень при запуске из папки Notebook.
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import compound, workspace, write_json
from rdkit import Chem,rdBase
from rdkit.Chem import Descriptors,rdMolDescriptors
OUT=workspace('M03-L01','case')
record=compound('istradefylline')
display(record)


{'name': 'istradefylline',
 'cid': 5311037,
 'smiles': 'CCN1C2=C(C(=O)N(C1=O)CC)N(C(=N2)/C=C/C3=CC(=C(C=C3)OC)OC)C',
 'formula': 'C20H24N4O4',
 'molecular_weight_g_mol': 384.4,
 'pubchem_XLogP': 2.5,
 'pubchem_TPSA_A2': 76.9,
 'source_url': 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/istradefylline/property/MolecularFormula,MolecularWeight,CanonicalSMILES,IsomericSMILES,XLogP,TPSA/JSON',
 'source_sha256': 'e74a82530b8e7096e37c1f893b42a60f454c86a805a4df59808961665ac2e80b',
 'property_kind': 'database representation/computed descriptor; not an assay or measured solubility'}

Я уже знаю, как закрепить входной файл. Теперь задача сложнее: убедиться, что после чтения химическая запись сохранила нужную информацию. Начать можно с имени или формулы, но эти признаки отвечают на разные вопросы. Имя помогает найти запись, формула — проверить состав; для следующего расчёта мне требуется определённый молекулярный граф с доступной стереохимией.

Поэтому я сохраняю исходную строку, а затем передаю её парсеру. Если парсер вернёт `None`, продолжать с дескрипторами нельзя: вместо молекулы у программы нет пригодного объекта. Если разбор успешен, я получаю граф в правилах выбранного инструмента и могу проверять, какие сведения он содержит. Именно эти сведения, а не успех самого вызова, определят следующее действие.

Словарь `record` хранит поля одной записи. Обращение по ключу `record['smiles']` выбирает конкретное представление. Передадим его химическому парсеру. Успешный синтаксический разбор не подтверждает, что реальный образец имеет этот состав.

In [3]:
raw_smiles=record['smiles']
mol=Chem.MolFromSmiles(raw_smiles)
if mol is None:
    raise ValueError('Источник не удалось прочитать как молекулярный граф')

# isomericSmiles=True сохраняет доступную информацию о конфигурации.
isomeric=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
connectivity=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=False)
display({'raw':raw_smiles,'canonical_isomeric':isomeric,'connectivity_only':connectivity})


{'raw': 'CCN1C2=C(C(=O)N(C1=O)CC)N(C(=N2)/C=C/C3=CC(=C(C=C3)OC)OC)C',
 'canonical_isomeric': 'CCn1c(=O)c2c(nc(/C=C/c3ccc(OC)c(OC)c3)n2C)n(CC)c1=O',
 'connectivity_only': 'CCn1c(=O)c2c(nc(C=Cc3ccc(OC)c(OC)c3)n2C)n(CC)c1=O'}

Почему я сохраняю две канонические строки? Канонизация выбирает согласованный способ записи графа: два разных порядка перечисления атомов не должны создавать две записи одной и той же структуры в рамках этого представления. Параметр `isomericSmiles` задаёт ещё один выбор — сохранять ли доступную стереоинформацию. Поэтому канонизация и стереохимическая полнота — две разные настройки политики сравнения.

Я могу получить одинаковую формулу и при этом потерять конфигурацию двойной связи. Следовательно, проверка состава полезна, но оставляет именно тот риск, который важен здесь: передать следующему расчёту менее определённое представление. Чтобы увидеть эту потерю на данных, сначала перечислю стереопризнаки связей, а затем выполню контролируемое упрощение.

До следующей ячейки назовите, какой символ в строке может относиться к геометрии двойной связи. Затем проверим признаки самого графа. `for` перебирает связи; список сохраняет наблюдения, которые позднее можно проверить.

In [4]:
bond_stereo=[]
for bond in mol.GetBonds():
    if bond.GetStereo()!=Chem.BondStereo.STEREONONE:
        bond_stereo.append({'atom_indices':[bond.GetBeginAtomIdx(),bond.GetEndAtomIdx()],
                            'stereo':str(bond.GetStereo())})

# Индекс атома — адрес в этом графе, не номер атома по химической номенклатуре.
summary={'cid':record['cid'],'formula':rdMolDescriptors.CalcMolFormula(mol),
         'molecular_weight_g_mol':Descriptors.MolWt(mol),
         'heavy_atoms':mol.GetNumHeavyAtoms(),'atoms_with_explicit_H':Chem.AddHs(mol).GetNumAtoms(),
         'double_bond_stereo':bond_stereo,'rdkit':rdBase.rdkitVersion}
display(summary)


{'cid': 5311037,
 'formula': 'C20H24N4O4',
 'molecular_weight_g_mol': 384.43600000000015,
 'heavy_atoms': 28,
 'atoms_with_explicit_H': 52,
 'double_bond_stereo': [{'atom_indices': [15, 16], 'stereo': 'STEREOE'}],
 'rdkit': '2026.09.1'}

Читаю полученную сводку в определённом порядке. Состав и масса показывают, что парсер создал согласованное представление по этим признакам. Число тяжёлых атомов описывает граф без водородов; добавленные явные H меняют способ представления, поэтому эти количества нельзя сравнивать как два независимых измерения состава. Стереопризнаки указывают, какая конфигурационная информация доступна для отдельных связей.

Теперь я хочу проверить ложный, но удобный критерий: «если формула и масса совпали, запись можно считать полностью сохранённой». Для этого изменю только способ сериализации стереоинформации и снова прочитаю граф. Такой контроль позволяет найти слабое место критерия, не обращаясь к выдуманному эксперименту.

Теперь проверим ловушку: упрощённая запись может утратить конфигурацию, сохранив массу и формулу. Поэтому совпадение двух чисел не заменяет выбранную политику идентичности.

In [5]:
simplified=Chem.MolFromSmiles(connectivity)
assert simplified is not None
assert rdMolDescriptors.CalcMolFormula(simplified)==summary['formula']
assert math.isclose(Descriptors.MolWt(simplified),summary['molecular_weight_g_mol'],abs_tol=1e-9)
assert Chem.MolToSmiles(simplified,isomericSmiles=True)!=isomeric
write_json(OUT/'identity_passport.json',{'source':record,'computed':summary,
          'canonical_isomeric':isomeric,'connectivity_only':connectivity,
          'conclusion':'Масса и формула не доказывают сохранение всей стереохимии',
          'next_check':'Уточнить соответствие экспериментального образца выбранной конфигурации'})
print(OUT/'identity_passport.json')


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\M03-L01\case\identity_passport.json


Результат определяет действие: connectivity-only строку можно хранить как отдельное представление для задач, которым она подходит, а для передачи конфигурационной информации сохраняем изомерную строку и исходную запись. Я не восстанавливаю конфигурацию из совпавшей массы: нужная информация в этом сравнении отсутствует.

В исследовании аторвастатина сначала придётся договориться ещё и о компонентах и заряде. Автоматическое удаление части записи может быть допустимой операцией стандартизации, но тогда в паспорте должны остаться исходный объект, правило преобразования и получившееся представление. Так следующий исследователь сможет проверить смысл перехода.

Мы обнаружили потерю информации при определённом преобразовании. Не установили активность, чистоту или свойства неизвестного образца. В досье переносится паспорт и ограничение. На следующем уроке проверим, что меняется при работе с солевой и лактонной формами.